# Durable Functions Client Test

This notebook sends a POST request to the local Azure Functions endpoint `http://localhost:7071/api/client` using the Python `requests` library.

Steps:
1. Define the JSON payload with a `records` array.
2. Send POST request; receive status URLs from Durable Functions (includes `statusQueryGetUri`).
3. Poll the status URL until the orchestration finishes.
4. Display final output.

Run the cells in order. Ensure the function host is running locally (e.g. via `func start` or your `startLocal.sh`).

In [1]:
# Define the payload matching sampleJSONrequest.json
payload = {
      "name": "role_library-3.pdf",
      "container": "bronze"
}
payload

{'name': 'role_library-3.pdf', 'container': 'bronze'}

In [2]:
import requests, json, time

FUNCTION_ENDPOINT = "http://localhost:7071/api/client"
# FUNCTION_ENDPOINT = "https://func-processing-yt7giyre5immc.azurewebsites.net/api/client?code="
# Send POST request to start orchestration
response = requests.post(FUNCTION_ENDPOINT, json=payload)
print("Status Code:", response.status_code)

if response.status_code != 202:
    print("Unexpected response:", response.text)
else:
    # Durable Functions returns JSON with status URLs
    start_info = response.json()
    # Display keys of interest
    for k in ["id", "statusQueryGetUri", "sendEventPostUri", "terminatePostUri", "purgeHistoryDeleteUri"]:
        print(f"{k}: {start_info.get(k)}")

print(start_info)

ConnectionError: HTTPConnectionPool(host='localhost', port=7071): Max retries exceeded with url: /api/client (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000173F5E40D70>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [ ]:
import requests, time, json

# Helper to poll the orchestration status until completion or timeout
TERMINAL_STATUSES = {"Completed", "Failed", "Terminated"}

def poll_status(status_url: str, interval_seconds: float = 2.0, timeout_seconds: float = 120.0):
    """Poll the Durable Functions statusQueryGetUri until a terminal status.
    Returns the final status document (dict) or raises TimeoutError.
    """
    start = time.time()
    attempt = 0
    while True:
        attempt += 1
        r = requests.get(status_url)
        if r.status_code != 200:
            print(f"Attempt {attempt}: Unexpected status code {r.status_code} -> {r.text[:200]}")
        else:
            status_doc = r.json()
            runtime_status = status_doc.get("runtimeStatus")
            print(f"Attempt {attempt}: runtimeStatus={runtime_status}")
            if runtime_status in TERMINAL_STATUSES:
                return status_doc
        if time.time() - start > timeout_seconds:
            raise TimeoutError(f"Polling exceeded {timeout_seconds} seconds without terminal status.")
        time.sleep(interval_seconds)

# Extract the status URL from the earlier cell output (start_info)
status_url = start_info.get("statusQueryGetUri")
if not status_url:
    raise ValueError("statusQueryGetUri not found in start_info. Run the previous cell first.")

final_status = poll_status(status_url)

print("\nFinal Status Document:\n")
print(json.dumps(final_status, indent=2))

# If Completed, the orchestrator's output is in 'output'
if final_status.get("runtimeStatus") == "Completed":
    print("\nOrchestrator Output:\n")
    print(json.dumps(final_status.get("output"), indent=2))
else:
    print("\nOrchestration did not complete successfully.")

## Voice-to-Voice Translation Test

This section tests the voice-to-voice translation functionality. 

**Prerequisites:**
1. Ensure `ENABLE_VOICE_TO_VOICE_TRANSLATION` is set to `"true"` in your function app settings
2. Ensure `AIMULTISERVICES_ENDPOINT` and `AIMULTISERVICES_RESOURCE_ID` are configured
3. Ensure `TRANSLATION_TARGET_LANGUAGE` is set (default: "es" for Spanish)
4. Upload an audio file (e.g., `hello.wav`) to the `bronze` container in blob storage

**Note:** For large files (>50MB), batch processing will be used automatically.

In [ ]:
# Test Voice-to-Voice Translation with hello.wav
# This will trigger the voiceToVoiceTranslation activity if ENABLE_VOICE_TO_VOICE_TRANSLATION is enabled

voice_translation_payload = {
    "name": "hello.wav",
    "container": "bronze"
}

print("Testing voice-to-voice translation with:", voice_translation_payload)
voice_translation_payload

In [ ]:
# Send POST request to start orchestration for voice translation
voice_response = requests.post(FUNCTION_ENDPOINT, json=voice_translation_payload)
print("Status Code:", voice_response.status_code)

if voice_response.status_code != 202:
    print("Unexpected response:", voice_response.text)
else:
    # Durable Functions returns JSON with status URLs
    voice_start_info = voice_response.json()
    # Display keys of interest
    for k in ["id", "statusQueryGetUri", "sendEventPostUri", "terminatePostUri", "purgeHistoryDeleteUri"]:
        print(f"{k}: {voice_start_info.get(k)}")

print("\nFull response:")
print(json.dumps(voice_start_info, indent=2))

In [ ]:
# Poll for voice translation orchestration completion
# Note: Voice translation can take longer, especially for batch processing
# Increase timeout for large files

voice_status_url = voice_start_info.get("statusQueryGetUri")
if not voice_status_url:
    raise ValueError("statusQueryGetUri not found in voice_start_info. Run the previous cell first.")

# Use longer timeout for voice translation (up to 30 minutes for batch processing)
try:
    voice_final_status = poll_status(voice_status_url, interval_seconds=5.0, timeout_seconds=1800.0)
    
    print("\nFinal Voice Translation Status:\n")
    print(json.dumps(voice_final_status, indent=2))
    
    # If Completed, the orchestrator's output is in 'output'
    if voice_final_status.get("runtimeStatus") == "Completed":
        print("\n=== Voice Translation Output ===\n")
        output = voice_final_status.get("output", {})
        print(json.dumps(output, indent=2))
        
        # Extract voice translation specific results
        if isinstance(output, dict):
            text_result = output.get("text_result")
            if text_result:
                print("\n=== Translated Text (first 500 chars) ===\n")
                print(text_result[:500] if len(text_result) > 500 else text_result)
    else:
        print("\nVoice translation orchestration did not complete successfully.")
        print(f"Runtime Status: {voice_final_status.get('runtimeStatus')}")
        if voice_final_status.get("output"):
            print("\nError details:")
            print(json.dumps(voice_final_status.get("output"), indent=2))
            
except TimeoutError as e:
    print(f"\nPolling timed out: {e}")
    print("For large files, batch processing may take 30+ minutes.")
    print("Check the function logs for progress.")

## Direct Activity Test (Optional)

If you want to test the voiceToVoiceTranslation activity directly (bypassing the orchestrator), 
you can use the following approach. This requires importing the activity function directly.

In [ ]:
# Direct activity test (requires local function environment)
# This cell tests the voiceToVoiceTranslation activity function directly
# Note: This requires the function app to be running locally and proper configuration

try:
    import sys
    import os
    
    # Add pipeline directory to path
    pipeline_path = os.path.join(os.getcwd(), 'pipeline')
    if pipeline_path not in sys.path:
        sys.path.insert(0, pipeline_path)
    
    # Import the activity function
    from activities import voiceToVoiceTranslation
    from configuration import Configuration
    
    # Test blob input
    test_blob_input = {
        "name": "hello.wav",
        "container": "bronze",
        "uri": None  # Will be constructed if not provided
    }
    
    print("Testing voiceToVoiceTranslation activity directly...")
    print(f"Input: {test_blob_input}")
    print("\nNote: This requires:")
    print("  - Function app running locally")
    print("  - Proper Azure credentials configured")
    print("  - AIMULTISERVICES_ENDPOINT and AIMULTISERVICES_RESOURCE_ID set")
    print("  - Audio file exists in blob storage")
    print("\nUncomment the line below to run the test:")
    print("# result = voiceToVoiceTranslation.run(test_blob_input)")
    
except ImportError as e:
    print(f"Import error: {e}")
    print("Make sure you're running this from the project root directory")
except Exception as e:
    print(f"Error: {e}")

Import error: No module named 'azure.durable_functions'
Make sure you're running this from the project root directory
